# AAT6153 Chapter 2 — Hands-on 2
## Sentiment Analysis with Embeddings (IMDB)
**Slides:** Hands-on 2 (31–36)

**Task:** predict whether a movie review is positive (1) or negative (0).
Two models are compared:
1. **Bag-of-words** (multi-hot) + Dense layers (baseline)
2. **Embedding** + GlobalAveragePooling1D + Dense layers

*Runtime: CPU works; a GPU (Runtime ▸ Change runtime type ▸ T4 GPU) is faster.*

---
## Step 1: Load the IMDB dataset (slide 32)

In [ ]:
import numpy as np
import keras
from keras import layers
from keras.datasets import imdb
from keras.utils import pad_sequences

keras.utils.set_random_seed(7)
vocab_size, maxlen = 10000, 200          # keep the 10,000 most frequent words
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)
print(len(x_train), len(x_test))         # 25000 25000
print("Positive reviews in training set:", y_train.mean())

## Step 2: Decode a review back into words (slide 32)
Keras IMDB convention: **0 = padding, 1 = start of review, 2 = out-of-vocabulary**; real words start at 3.

In [ ]:
word_index = imdb.get_word_index()        # word -> rank
index_word = {i + 3: w for w, i in word_index.items()}
index_word.update({0: "<pad>", 1: "<start>", 2: "<oov>"})

print(" ".join(index_word.get(i, "?") for i in x_train[0][:25]))
print("Label:", y_train[0])               # 1 = positive, 0 = negative

In [ ]:
# Reviews have very different lengths (slide 4), which is why we pad / truncate to maxlen = 200
import matplotlib.pyplot as plt
lengths = [len(r) for r in x_train]
plt.hist(lengths, bins=60); plt.axvline(maxlen, color="r", ls="--", label=f"maxlen = {maxlen}")
plt.xlabel("Review length (tokens)"); plt.ylabel("Number of reviews"); plt.legend(); plt.show()
print(f"Median length: {int(np.median(lengths))},  longer than {maxlen}: {np.mean(np.array(lengths) > maxlen):.0%}")

---
## Step 3: Train a bag-of-words baseline (slide 33)
Each review becomes a 10,000-dimensional multi-hot vector: 1 if a word appears anywhere in it, so **word order is lost**.

In [ ]:
def multi_hot(sequences, dim=vocab_size):
    out = np.zeros((len(sequences), dim), dtype="float32")
    for i, seq in enumerate(sequences):
        out[i, seq] = 1.0                 # 1 if the word appears
    return out

xb_train, xb_test = multi_hot(x_train), multi_hot(x_test)

bow_model = keras.Sequential([
    keras.Input(shape=(vocab_size,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")])
bow_model.compile(optimizer="rmsprop", loss="binary_crossentropy",
                  metrics=["accuracy"])
bow_model.fit(xb_train, y_train, epochs=4, batch_size=512,
              validation_split=0.2)
print(bow_model.evaluate(xb_test, y_test))

---
## Step 4: Define an embedding-based model (slide 34)
Each review is padded / truncated to 200 tokens. The `Embedding` layer turns it into a 200 × 32 matrix, and `GlobalAveragePooling1D` averages the 200 word vectors into one 32-dimensional review vector.

In [ ]:
x_train_p = pad_sequences(x_train, maxlen=maxlen)
x_test_p  = pad_sequences(x_test,  maxlen=maxlen)

model = keras.Sequential([
    keras.Input(shape=(maxlen,)),
    layers.Embedding(vocab_size, 32),     # (batch, 200) -> (batch, 200, 32)
    layers.GlobalAveragePooling1D(),      # average over the 200 words
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")])
model.summary()

**Check:** Embedding 10,000 × 32 = 320,000, Dense 32 × 16 + 16 = 528, Dense 16 + 1 = 17 → **320,545** parameters (slide 35).

## Step 5: Train with early stopping and evaluate (slide 35)

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                   restore_best_weights=True)
history = model.fit(x_train_p, y_train, epochs=20, batch_size=128,
                    validation_split=0.2, callbacks=[es])
loss, acc = model.evaluate(x_test_p, y_test)
print(f"Test accuracy: {acc:.2%}")

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history, title):
    h = history.history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(h["accuracy"], "b", label="train"); ax1.plot(h["val_accuracy"], "g", label="val")
    ax1.set(xlabel="Epoch", ylabel="Accuracy", title=f"{title}: accuracy"); ax1.legend()
    ax2.plot(h["loss"], "b", label="train"); ax2.plot(h["val_loss"], "g", label="val")
    ax2.set(xlabel="Epoch", ylabel="Loss", title=f"{title}: loss"); ax2.legend()
    plt.show()

plot_history(history, "Embedding + average pooling")

In [ ]:
# Compare the two models (record these numbers for Hands-on 4)
bow_acc = bow_model.evaluate(xb_test, y_test, verbose=0)[1]
print(f"{'Model':40s} Test accuracy")
print(f"{'Bag-of-words + Dense':40s} {bow_acc:.2%}")
print(f"{'Embedding + average pooling':40s} {acc:.2%}")

---
## Step 6: Inspect the learned embeddings (slide 36)
After training on sentiment labels, the embedding matrix encodes sentiment: the nearest neighbours of a positive word tend to be other positive words. Results vary from run to run.

In [ ]:
weights = model.layers[0].get_weights()[0]       # (10000, 32)

def nearest(word, k=5):
    v = weights[word_index[word] + 3]
    sims = weights @ v / (np.linalg.norm(weights, axis=1)
                          * np.linalg.norm(v) + 1e-8)
    best = np.argsort(sims)[::-1][1:k + 1]
    return [index_word.get(i, "?") for i in best]

print(nearest("great"))
print(nearest("awful"))

---
## Try it yourself (slide 36)
1. Change the embedding size to **8** and then **128**. What happens to the accuracy and the number of parameters?
2. Replace `GlobalAveragePooling1D()` with `Flatten()`. How many parameters does the next Dense layer have now? Does accuracy change?
3. Find the nearest neighbours of other words, e.g. `"boring"`, `"masterpiece"`, `"waste"`.

In [ ]:
# Use this cell for your experiments